In [1]:
YOUR_NAME = 'sara'

AWS_PROFILE = 'cities'

In [2]:
MAIN_PATH = "s3://wri-cities-sandbox/identifyingLandSubdivisions/data"
INPUT_PATH = f'{MAIN_PATH}/input'
CITY_INFO_PATH = f'{INPUT_PATH}/city_info'
EXTENTS_PATH = f'{CITY_INFO_PATH}/extents'
BUILDINGS_PATH = f'{INPUT_PATH}/buildings'
BLOCKS_PATH = f'{INPUT_PATH}/blocks'
ROADS_PATH = f'{INPUT_PATH}/roads'
INTERSECTIONS_PATH = f'{INPUT_PATH}/intersections'
GRIDS_PATH = f'{INPUT_PATH}/city_info/grids'
SEARCH_BUFFER_PATH = f'{INPUT_PATH}/city_info/search_buffers'
OUTPUT_PATH = f'{MAIN_PATH}/output'
OUTPUT_PATH_CSV = f'{OUTPUT_PATH}/csv'
OUTPUT_PATH_RASTER = f'{OUTPUT_PATH}/raster'
OUTPUT_PATH_PNG = f'{OUTPUT_PATH}/png'
OUTPUT_PATH_RAW = f'{OUTPUT_PATH}/raw_results'

In [3]:
# Check s3 connection using AWS_PROFILE=CitiesUserPermissionSet profile 
import boto3
boto3.setup_default_session(profile_name='cities')
session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client('s3')

# export CitiesUserPermissionSet profile to use in the next cells
import os
os.environ['AWS_PROFILE'] = AWS_PROFILE

s3.list_buckets()

{'ResponseMetadata': {'RequestId': 'S9P13Z8H92BAVFP9',
  'HostId': '5b1gZyB4KOK6AAh52/jF/prvoNAypBN5bnaIWv5fSjwNgJ1JYauZzEYfMeCh9Mr3NxYCWGbTDCU=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '5b1gZyB4KOK6AAh52/jF/prvoNAypBN5bnaIWv5fSjwNgJ1JYauZzEYfMeCh9Mr3NxYCWGbTDCU=',
   'x-amz-request-id': 'S9P13Z8H92BAVFP9',
   'date': 'Sun, 29 Mar 2026 00:50:06 GMT',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'aft-sandbox-540362055257',
   'CreationDate': datetime.datetime(2022, 9, 13, 15, 12, 20, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::aft-sandbox-540362055257'},
  {'Name': 'amplify-citiesindicatorsapi-dev-10508-deployment',
   'CreationDate': datetime.datetime(2023, 8, 30, 5, 5, 13, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::amplify-citiesindicatorsapi-dev-10508-deployment'},
  {'Name': 'cities-heat',
   'CreationDate': datetime.datetime(2023, 6, 1, 13, 22, 1, tzinfo=tzutc

In [4]:
import dask_geopandas as dgpd
import dask.dataframe as dd
import pandas as pd
from dask import delayed, compute, visualize
import geopandas as gpd
from dask.diagnostics import ProgressBar
from shapely.geometry import MultiLineString, LineString, Point
from shapely.ops import polygonize, nearest_points
#from shapely.geometry import Polygon, LineString, Point, MultiPolygon, MultiLineString, GeometryCollection
from scipy.optimize import fminbound, minimize
#from unused_code.metrics_groupby import metrics
from dask import delayed
import dask_geopandas as dgpd
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import wkb
from scipy.stats import entropy
import time
from dask import compute


from pre_processing import *
from auxiliary_functions import *
from standardize_metrics import *
from metrics_calculation import *

import shutil

import time

YOUR_NAME = 'sara'

In [5]:
def read_validation_set_s3(bucket_prefix = "wri-cities-sandbox/identifyingLandSubdivisions/data/validation/Land_Use/102_cities",
                           base = "Final_102Cities"):
    
    # or use 9_cities/Sub_Saharan_9_cities_and_LAC_3_cities_Land_use
    
    fs = s3fs.S3FileSystem(anon=False)

    # All components of the shapefile
    suffixes = [".shp", ".dbf", ".shx", ".prj", ".cpg"]

    local_dir = "/tmp/land_use"
    os.makedirs(local_dir, exist_ok=True)

    for suf in suffixes:
        s3_path = f"{bucket_prefix}/{base}{suf}"
        local_path = os.path.join(local_dir, base + suf)
        with fs.open(s3_path, "rb") as fsrc, open(local_path, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)

    # Now read from local path
    local_shp = os.path.join(local_dir, base + ".shp")
    validation_set = gpd.read_file(local_shp)

    return validation_set

validation_set = read_validation_set_s3()

mix = "Democratic_Republic_of_the_Congo_Republic_of_Congo"

# Brazzaville -> Kinshasa
validation_set["City"] = validation_set["City"].mask(validation_set["City"].eq("Brazzaville"), "Kinshasa")

# Set Country to the mixed one for (original) Kinshasa and Brazzaville rows
validation_set["Country"] = validation_set["Country"].mask(
    validation_set["City"].eq("Kinshasa"),  # after the line above, Brazzaville is now Kinshasa too
    mix
)

validation_set.loc[validation_set["City"] == "N_Djamena", "Country"] = "Chad"

cities = validation_set.City.unique()

validation_set[validation_set.Country.isna()]


MAIN_PATH = "s3://wri-cities-sandbox/identifyingLandSubdivisions/data"
OUTPUT_PATH = f"{MAIN_PATH}/output"
OUTPUT_PATH_RASTER = f"{OUTPUT_PATH}/raster"


repl = {
        "United Republic of Tanzania": "Tanzania",
        "Congo": "Republic of Congo",
        "Pointe-Noire": "Pointe Noire",
        "Côte d'Ivoire": "Cote d Ivoire",
        "CÃ´te d'Ivoire": "Cote d Ivoire",
        "Cã´Te-D'Ivoire" : "Cote d Ivoire",
        "Mbour": "MBour",
        "Manzin": "Manzini",
        "Burkina-Faso":"Burkina Faso",
        "Sierra-Leone":"Sierra Leone",
        "South-Africa": "South Africa",
        "Guinea-Bissau": "Guinea Bissau",
        "Ido ekiti":"Ido Ekiti",
        'Port-gentil':"Port Gentil"
            }

validation_set[["City", "Country"]] = validation_set[["City", "Country"]].replace(repl)


cities = validation_set[['City','Country']].drop_duplicates().apply(lambda x: "_".join(x['City'].split())+'__'+"_".join(x['Country'].split()), axis=1).values 

cities = list(set(cities) - set(['Banki__Nigeria', 'Maseru__South_Africa', 'Lome__Ghana','N_Djamena__Cameroon']))

'''
OR ANOTHER SET OF CITIES
'''

'\nOR ANOTHER SET OF CITIES\n'

In [7]:
paths

['s3://wri-cities-sandbox/identifyingLandSubdivisions/data/output/raster/Koidu_Town__Sierra_Leone/Koidu_Town__Sierra_Leone_block_metrics_ALL_sara.geoparquet',
 's3://wri-cities-sandbox/identifyingLandSubdivisions/data/output/raster/Yaounde__Cameroon/Yaounde__Cameroon_block_metrics_ALL_sara.geoparquet',
 's3://wri-cities-sandbox/identifyingLandSubdivisions/data/output/raster/Niamey__Niger/Niamey__Niger_block_metrics_ALL_sara.geoparquet',
 's3://wri-cities-sandbox/identifyingLandSubdivisions/data/output/raster/Bissau__Guinea_Bissau/Bissau__Guinea_Bissau_block_metrics_ALL_sara.geoparquet',
 's3://wri-cities-sandbox/identifyingLandSubdivisions/data/output/raster/Antananarivo__Madagascar/Antananarivo__Madagascar_block_metrics_ALL_sara.geoparquet',
 's3://wri-cities-sandbox/identifyingLandSubdivisions/data/output/raster/Wa__Ghana/Wa__Ghana_block_metrics_ALL_sara.geoparquet',
 's3://wri-cities-sandbox/identifyingLandSubdivisions/data/output/raster/Masvingo__Zimbabwe/Masvingo__Zimbabwe_block_m

In [8]:
gdfs

[]

In [9]:

paths = [f'{OUTPUT_PATH_RASTER}/{city}/{city}_block_metrics_ALL_{YOUR_NAME}.geoparquet' 
         for city in cities]

gdfs = []
# Read into GeoPandas, reproject to common CRS, and concat
for path in paths:
    gdf_temp = gpd.read_parquet(path).to_crs("EPSG:4326")
    gdf_temp['City'] = path.split('/')[-2].split('__')[0]
    gdf_temp['Country'] = path.split('/')[-2].split('__')[1]
    
consolidated_cities = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs="EPSG:4326")


KeyboardInterrupt: 

In [ ]:

def consolidate_irregularity_index(consolidated_cities):
    # 1) Paths (same as before) …
    
    df = consolidated_cities.copy()


    df['has_features'] = df['has_buildings'] | df['has_roads'] | df['has_intersections']

    all_metrics_columns_final = ['m'+str(x)+'_final' for x in range(1,13)]
    all_metrics_columns_std = ['m'+str(x)+'_std' for x in range(1,13)]

    
    # 6) Compute global scalars on standardized metrics
    means = df[all_metrics_columns_std].mean()
    stds  = df[all_metrics_columns_std].std()
    mins  = df[all_metrics_columns_std].min()
    maxs  = df[all_metrics_columns_std].max()


    df[all_metrics_columns_final] = (df[all_metrics_columns_std] - df[all_metrics_columns_std].min()) / (df[all_metrics_columns_std].max() - df[all_metrics_columns_std].min())

    
    '''
    I'll have to actually consolidate the index here, once i have the weights.
    '''

    out = f'{OUTPUT_PATH}/consolidated_cities_results.geoparquet'
    # … after your df = df.map_partitions(...), instead of WKB+p

    # write **one** parquet file, with embedded CRS and GeoArrow extension types:
    df.to_parquet(out, engine='pyarrow', index=False)

    return out

consolidated_cities_with_index = consolidate_irregularity_index(consolidated_cities)
